# Exercise 11.2: Central Limit Theorem and Bootstrap

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/Maths-for-IT-Notebooks/blob/main/Chapter-11/02-central-limit-theorem-bootstrap.ipynb)

**Book Reference:** Chapter 11 — Probability and Statistical Inference (`chapters/12-probability-and-statistical-inference.tex`, Sections 2–3)  
**Estimated Time:** 20–25 min

### Learning Objectives
- Generate an empirical sampling distribution of sample proportions $\widehat p$ across repeated samples.
- Compare empirical standard deviation with theoretical standard error $\operatorname{SE}(\widehat p) = \sqrt{p(1-p)/n}$.
- Compute a non-parametric percentile bootstrap confidence interval from a single observed sample.


In [ ]:
import math
import numpy as np

### Mathematical Context

Under independent Bernoulli($p$) sampling of size $n$, the sample proportion $\widehat p$ has expected value $p$ and theoretical standard error:
$$\operatorname{SE}(\widehat p) = \sqrt{\frac{p(1-p)}{n}}.$$

When $n$ is sufficiently large, the sampling distribution of $\widehat p$ is approximately Normal centred at $p$ with standard deviation $\operatorname{SE}(\widehat p)$ (Central Limit Theorem).
In practice, we only observe **one** sample. **Bootstrapping** resamples with replacement from that observed dataset to approximate the sampling distribution and construct an empirical confidence interval.

In [ ]:
def theoretical_se(p: float, n: int) -> float:
    """Compute theoretical standard error of a sample proportion.

    Formula:
        SE(p_hat) = sqrt(p * (1 - p) / n)

    Args:
        p: Population proportion in [0.0, 1.0].
        n: Sample size (n > 0).

    Returns:
        Theoretical standard error as a float.
    """
    # TODO: Validate p and n, then return math.sqrt(p * (1.0 - p) / n).
    raise NotImplementedError("Complete this TODO")


def sampling_distribution_means(
    p: float, n: int, num_samples: int = 2000, seed: int = 42
) -> np.ndarray:
    """Simulate num_samples independent samples of size n from Bernoulli(p).

    Args:
        p: True population proportion in [0.0, 1.0].
        n: Sample size per draw (n > 0).
        num_samples: Number of repeated samples to simulate.
        seed: Random seed (default 42).

    Returns:
        1D NumPy array of shape (num_samples,) containing sample proportions.
    """
    # TODO: Use rng = np.random.default_rng(seed) and rng.binomial(n, p, size=num_samples) / n.
    raise NotImplementedError("Complete this TODO")


def bootstrap_proportion_ci(
    data: np.ndarray, num_resamples: int = 2000, alpha: float = 0.05, seed: int = 42
) -> tuple[float, float, float]:
    """Compute a percentile bootstrap confidence interval for a binary sample.

    Args:
        data: 1D NumPy array of binary outcomes (0/1 or bool).
        num_resamples: Number of bootstrap resamples with replacement.
        alpha: Significance level (default 0.05 for 95% CI).
        seed: Random seed (default 42).

    Returns:
        Tuple (p_hat, ci_low, ci_high) where p_hat is np.mean(data) and
        ci_low, ci_high are the (alpha/2) and (1 - alpha/2) percentiles.
    """
    # TODO: Resample indices or values with replacement using rng.integers(0, len(data), size=(num_resamples, len(data)))
    # and compute np.quantile on the resample means.
    raise NotImplementedError("Complete this TODO")

<details>
<summary>Hint 1: Fast vectorized sampling distribution</summary>

In NumPy, `rng.binomial(n=n, p=p, size=num_samples)` draws `num_samples` Binomial($n, p$) counts in a single vectorized call. Divide by `n` to get the sample proportions.

</details>

<details>
<summary>Hint 2: Vectorized bootstrap resampling</summary>

Generate an index matrix of shape `(num_resamples, n)` via `rng.integers(0, n, size=(num_resamples, n))`, index into your 1D `data` array, and take `np.mean(..., axis=1)`. Then use `np.quantile(boot_means, alpha / 2.0)` and `np.quantile(boot_means, 1.0 - alpha / 2.0)`.

</details>

In [ ]:
# Self-check assertions
se_ctr = theoretical_se(0.30, 400)
assert abs(se_ctr - 0.022912878) < 1e-6, f"Unexpected SE: {se_ctr}"

samples = sampling_distribution_means(p=0.30, n=400, num_samples=5000, seed=42)
assert samples.shape == (5000,)
assert abs(float(np.mean(samples)) - 0.30) < 0.005
assert abs(float(np.std(samples, ddof=1)) - se_ctr) < 0.002

# Bootstrap on the exact 120/400 click-through sample
ctr_data = np.array([1] * 120 + [0] * 280, dtype=float)
p_hat, lo, hi = bootstrap_proportion_ci(ctr_data, num_resamples=2000, alpha=0.05, seed=42)
assert abs(p_hat - 0.30) < 1e-12
assert 0.24 < lo < 0.27 and 0.33 < hi < 0.36

print(f"Theoretical SE (p=0.30, n=400): {se_ctr:.5f}")
print(f"Empirical SD of 5000 samples:   {np.std(samples, ddof=1):.5f}")
print(f"Bootstrap 95% CI: [{lo:.3f}, {hi:.3f}] (Normal approx: [0.255, 0.345])")
print("All assertions passed!")

### Think Deeper

1. Why does the bootstrap resample **with replacement** rather than without replacement? What would happen to every resample mean if we resampled $n$ items without replacement?
2. Compare the bootstrap 95% CI on `ctr_data` ($120/400$) with the normal-approximation interval $[0.255, 0.345]$. When might the two methods disagree noticeably?